# Pré-processamento — Dataset CSEDM / ProgSnap2 v6

Transforma os dados brutos do CSEDM em sequências KT prontas para BKT, DKT e Code-DKT.  
Metodologia: fase de *Data Preparation* do EDM Process (Kalita et al., 2025).

---

## Splits do Dataset

O CSEDM disponibiliza dois semestres independentes, cada um com Train/Test próprios:

| Split | Semestre | Estudantes | Uso recomendado |
|-------|----------|------------|-----------------|
| `All/` | Fall-2019 (set–dez) | 506 | EDA exploratória completa |
| `Release/` | Spring-2019 (fev–mai) | 329 | **Comparação reproduzível com Shi et al. (2022)** |

As populações **não se sobrepõem**: SubjectIDs de `All/` ≠ SubjectIDs de `Release/`.  
Para replicar os resultados do paper Code-DKT (AUC ~74% em A1), usar sempre `Release/Train` para treino e `Release/Test` para teste.  
O benchmark de reprodutibilidade é **23.70% de tentativas corretas em Release/Train** (Shi et al. (2022) reporta 23.68% — margem de arredondamento esperada).

**Referência:** Shi et al. (2022); Pankiewicz, Shi & Baker (2025).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

NOTEBOOK_DIR = Path().resolve()
ROOT = NOTEBOOK_DIR.parent
DATA_ROOT = ROOT / 'data' / 'CSEDM'
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

# Adicionar src/ ao path para importar data_loader
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_loader import load_main_table, load_labels

print(f'SEED = {SEED}')
print(f'DATA_ROOT = {DATA_ROOT}')
print(f'DATA_ROOT existe: {DATA_ROOT.exists()}')

SEED = 42
DATA_ROOT = /home/leokuntz/Documents/repositories/studies/tcc.edm.kt/data/CSEDM
DATA_ROOT existe: True


---

## 1 — Setup e Carregamento dos Dados

### 1.1 — Carregamento dos splits

**Contexto:** O primeiro passo de qualquer pipeline de pré-processamento é carregar os dados e confirmar que os splits têm as dimensões esperadas. Aqui carregamos os quatro splits relevantes: `All/Data` (EDA completa), `Release/Train` e `Release/Test` (treinamento e avaliação reproduzíveis conforme o protocolo de Shi et al. (2022)).  
**Hipótese:** `All/Data` deve conter ~360 mil eventos e 506 estudantes. `Release/Train` deve ter ~134 mil eventos com 23.70% de corretos em `Run.Program`. `Release/Test` deve ter ~32 mil eventos e 83 estudantes.  
**Referência:** Price et al. (2020); Shi et al. (2022).

In [2]:
# Carregar todos os splits
all_main        = load_main_table('all',           DATA_ROOT)
release_train   = load_main_table('release_train', DATA_ROOT)
release_test    = load_main_table('release_test',  DATA_ROOT)

# Labels (predição final do pipeline KT)
early_train = load_labels('release_train', DATA_ROOT, which='early')
late_train  = load_labels('release_train', DATA_ROOT, which='late')
early_test  = load_labels('release_test',  DATA_ROOT, which='early')
late_test   = load_labels('release_test',  DATA_ROOT, which='late')

splits = {
    'All/Data':        all_main,
    'Release/Train':   release_train,
    'Release/Test':    release_test,
}

print(f'{"Split":<20} {"Eventos":>10} {"Estudantes":>12} {"Assignments":>13}')
print('-' * 58)
for name, df in splits.items():
    n_events    = len(df)
    n_subjects  = df['SubjectID'].nunique()
    n_assign    = df['AssignmentID'].nunique()
    print(f'{name:<20} {n_events:>10,} {n_subjects:>12,} {n_assign:>13,}')

Split                   Eventos   Estudantes   Assignments
----------------------------------------------------------
All/Data                360,176          506             5
Release/Train           134,508          246             5
Release/Test             32,372           83             3


**Achado:** Os splits têm as dimensões esperadas — `All/Data` com ~360k eventos e 506 estudantes; `Release/Train` com ~134k e 246 estudantes; `Release/Test` com ~32k e 83 estudantes. Populações sem sobreposição (semestres distintos).  
**Implicação para modelagem:** Toda comparação com os resultados de Shi et al. (2022) deve usar exclusivamente `Release/Train` e `Release/Test`. O split `All/` é reservado para análises exploratórias e não deve ser misturado com `Release/`.

### 1.2 — Benchmark de reprodutibilidade

**Contexto:** Shi et al. (2022) reporta que o dataset de treinamento contém 23.68% de tentativas corretas (Run.Program com Score == 1.0). Confirmar este valor é o primeiro teste de reprodutibilidade do pipeline.  
**Hipótese:** A taxa calculada em `Release/Train` deve ser ~23.68–23.70%, refletindo apenas diferença de arredondamento entre os papers.  
**Referência:** Shi et al. (2022) — Section 4.1, "23.68% of the attempts are correct".

In [3]:
runs_train = release_train[release_train['EventType'] == 'Run.Program'].copy()
runs_test  = release_test[release_test['EventType']  == 'Run.Program'].copy()

correct_rate_train = (runs_train['Score'] == 1.0).mean()
correct_rate_test  = (runs_test['Score']  == 1.0).mean()

print(f'Release/Train — Run.Program: {len(runs_train):,} eventos')
print(f'  Taxa de corretos (Score == 1.0): {correct_rate_train:.4%}')
print(f'  Paper Shi et al. (2022):          23.6800%')
print(f'  Divergência:                      {abs(correct_rate_train - 0.2368):.4%}')
print()
print(f'Release/Test — Run.Program: {len(runs_test):,} eventos')
print(f'  Taxa de corretos (Score == 1.0): {correct_rate_test:.4%}')

Release/Train — Run.Program: 46,825 eventos
  Taxa de corretos (Score == 1.0): 23.7010%
  Paper Shi et al. (2022):          23.6800%
  Divergência:                      0.0210%

Release/Test — Run.Program: 10,845 eventos
  Taxa de corretos (Score == 1.0): 21.1803%


**Achado:** A taxa de corretos em `Release/Train` é ~23.70%, a menos de 0.02pp do valor de 23.68% reportado por Shi et al. (2022) — diferença de arredondamento esperada. O benchmark de reprodutibilidade é confirmado.  
**Implicação para modelagem:** O dataset está íntegro para reprodução do paper. A baixa proporção de acertos (~23.7%) justifica o uso de AUC como métrica primária em vez de acurácia — uma predição trivial "sempre errado" teria acurácia de 76.3%, mas AUC de ~50%.

### 1.3 — Estrutura e tipos de dados

**Contexto:** Verificar se as colunas críticas estão presentes e com tipos corretos. Os modelos KT dependem de `SubjectID`, `AssignmentID`, `ProblemID`, `ServerTimestamp` (ordem cronológica), `EventType` e `Score`.  
**Hipótese:** Todas as colunas críticas devem estar presentes em `Release/Train`. O `Release/` tem uma coluna extra `CourseSectionID` e `SourceLocation` ausentes em `All/`.  
**Referência:** Price et al. (2020) — ProgSnap2 v6 specification.

In [4]:
CRITICAL_COLS = ['SubjectID', 'AssignmentID', 'ProblemID', 'ServerTimestamp',
                 'EventType', 'Score', 'CodeStateID']

print('=== Release/Train — dtypes e nulos ===')
info = pd.DataFrame({
    'dtype':  release_train.dtypes.astype(str),
    'nulls':  release_train.isna().sum(),
    'null_%': (release_train.isna().mean() * 100).round(2),
})
print(info.to_string())
print()

missing = [c for c in CRITICAL_COLS if c not in release_train.columns]
if missing:
    print(f'AVISO — colunas críticas ausentes: {missing}')
else:
    print('OK — todas as colunas críticas presentes em Release/Train')

# Score só existe em Run.Program — nulo em Compile e Compile.Error é esperado
score_null_outside_run = release_train[
    (release_train['EventType'] != 'Run.Program') & release_train['Score'].notna()
]
print(f'Score não-nulo fora de Run.Program: {len(score_null_outside_run)} (esperado: 0)')

=== Release/Train — dtypes e nulos ===
                                         dtype  nulls  null_%
Order                                    int64      0    0.00
SubjectID                                  str      0    0.00
ToolInstances                              str      0    0.00
ServerTimestamp            datetime64[us, UTC]      0    0.00
ServerTimezone                             str      0    0.00
CourseID                                   str      0    0.00
CourseSectionID                          int64      0    0.00
AssignmentID                             Int64      0    0.00
ProblemID                                Int64      0    0.00
CodeStateID                                str      0    0.00
IsEventOrderingConsistent                 bool      0    0.00
EventType                                  str      0    0.00
Score                                  float64  87683   65.19
Compile.Result                             str  87683   65.19
CompileMessageType             

**Achado:** Todas as colunas críticas estão presentes. `Score` é nulo em `Compile` e `Compile.Error` (comportamento esperado — Score só existe em `Run.Program`). `ServerTimestamp` foi convertido para `datetime64[ns, UTC]` pelo `load_main_table`, garantindo ordenação cronológica correta.  
**Implicação para modelagem:** Nenhuma engenharia de colunas adicional é necessária para o carregamento. O `load_main_table` de `src/data_loader.py` entrega o DataFrame pronto para as etapas de filtragem (tarefa 2).

---

## 2 — Filtragem por Modelo

### 2.1 — filter_for_bkt_dkt e filter_for_code_dkt

**Contexto:** BKT e DKT processam apenas sequências de acerto/erro sem informação do código-fonte; portanto, entram na sequência somente eventos `Run.Program` (execuções com Score). Code-DKT, por outro lado, extrai embeddings AST a cada submissão — incluindo código com erros de compilação — e por isso adiciona os eventos `Compile.Error` como `correct=0`. Essa distinção de protocolo é central para reproduzir os resultados de Shi et al. (2022) e Pankiewicz, Shi & Baker (2025).  
**Hipótese:** Após a filtragem, BKT/DKT devem conter apenas `Run.Program` (~46k eventos em Release/Train); Code-DKT deve conter `Run.Program` + `Compile.Error` (~87k eventos). A proporção de corretos em Code-DKT deve ser menor que em BKT/DKT, pois os `Compile.Error` contribuem exclusivamente com `correct=0`.  
**Referência:** Shi et al. (2022) — Section 3.1, "correct=1 when all tests pass"; Pankiewicz, Shi & Baker (2025) — srcML-DKT, EDM 2025.

In [5]:
from data_loader import filter_for_bkt_dkt, filter_for_code_dkt

# --- Filtragem ---
bkt_dkt_train = filter_for_bkt_dkt(release_train)
bkt_dkt_test  = filter_for_bkt_dkt(release_test)

code_dkt_train = filter_for_code_dkt(release_train)
code_dkt_test  = filter_for_code_dkt(release_test)

# --- Verificação de EventType: nenhum outro tipo deve ter passado ---
assert set(bkt_dkt_train['EventType'].unique()) == {'Run.Program'}, "ERRO: EventType inesperado em bkt_dkt_train"
assert set(bkt_dkt_test['EventType'].unique())  == {'Run.Program'}, "ERRO: EventType inesperado em bkt_dkt_test"
assert set(code_dkt_train['EventType'].unique()).issubset({'Run.Program', 'Compile.Error'}), "ERRO: EventType inesperado em code_dkt_train"
assert set(code_dkt_test['EventType'].unique()).issubset({'Run.Program', 'Compile.Error'}),  "ERRO: EventType inesperado em code_dkt_test"
print("OK — assertions de EventType passaram em todos os splits")

# --- Estatísticas dos filtros ---
print()
print(f"{'Split':<30} {'Eventos':>10} {'EventTypes':>30} {'Taxa correto':>14}")
print('-' * 90)

for name, df in [
    ('BKT/DKT — Release/Train',   bkt_dkt_train),
    ('BKT/DKT — Release/Test',    bkt_dkt_test),
    ('Code-DKT — Release/Train',  code_dkt_train),
    ('Code-DKT — Release/Test',   code_dkt_test),
]:
    event_types = ', '.join(sorted(df['EventType'].unique()))
    correct_rate = df['correct'].mean()
    print(f"{name:<30} {len(df):>10,} {event_types:>30} {correct_rate:>14.4%}")

# --- Detalhe de EventType em Code-DKT ---
print()
print("Code-DKT — Release/Train: contagem por EventType")
print(code_dkt_train['EventType'].value_counts().to_string())
print()
print("Code-DKT — Release/Test: contagem por EventType")
print(code_dkt_test['EventType'].value_counts().to_string())

# --- Confirmar que colunas 'correct' são binárias ---
assert bkt_dkt_train['correct'].isin([0, 1]).all(),  "correct não é binário em bkt_dkt_train"
assert code_dkt_train['correct'].isin([0, 1]).all(), "correct não é binário em code_dkt_train"
print()
print("OK — coluna 'correct' é binária (0/1) em ambos os filtros")

OK — assertions de EventType passaram em todos os splits

Split                             Eventos                     EventTypes   Taxa correto
------------------------------------------------------------------------------------------


BKT/DKT — Release/Train            46,825                    Run.Program       23.7010%
BKT/DKT — Release/Test             10,845                    Run.Program       21.1803%
Code-DKT — Release/Train           87,683     Compile.Error, Run.Program       12.6570%
Code-DKT — Release/Test            21,527     Compile.Error, Run.Program       10.6703%

Code-DKT — Release/Train: contagem por EventType
EventType
Run.Program      46825
Compile.Error    40858

Code-DKT — Release/Test: contagem por EventType
EventType
Run.Program      10845
Compile.Error    10682

OK — coluna 'correct' é binária (0/1) em ambos os filtros


**Achado:** `filter_for_bkt_dkt` retém apenas `Run.Program` — 46.825 eventos em Release/Train e 10.845 em Release/Test, com taxa de corretos de 23.70% e 21.18%, respectivamente. `filter_for_code_dkt` adiciona os `Compile.Error`, elevando o total para 87.683 eventos em Train (Run.Program: 46.825 + Compile.Error: 40.858), taxa de corretos de 12.66% — os Compile.Error diluem a proporção pois contribuem somente com `correct=0`. Todas as assertions de EventType passam: nenhum outro tipo de evento (e.g., `Compile`) escapou pelos filtros; `correct` é binária (0/1) em ambos os casos.  
**Implicação para modelagem:** As sequências BKT/DKT partem de 23.70% de corretos; Code-DKT parte de 12.66% — o imbalance do Code-DKT é maior porque inclui os 40.858 Compile.Error como incorretos. A métrica AUC (e não acurácia) é essencial em ambos os cenários. A decisão de incluir Compile.Error no Code-DKT é empírica e teórica: empiricamente, `n_compile_errors` tem Spearman |ρ|=0.569 com o Label (Seção 8 do EDA); teoricamente, esses eventos carregam informação da evolução do código que o srcML consegue parsear (Pankiewicz et al., 2025).

---

## 3 — Construção de Sequências KT

### 3.1 — build_sequences()

**Contexto:** Os modelos BKT, DKT e Code-DKT recebem como entrada uma sequência de tentativas de um estudante em um assignment, ordenada cronologicamente. A função `build_sequences` transforma o DataFrame filtrado em listas de sequências per-student, adicionando a flag `is_first_attempt` necessária para calcular o *first-attempt AUC* — a métrica primária de Shi et al. (2022). Processar por assignment é essencial porque os modelos são treinados independentemente para cada assignment (protocolo KC = ProblemID por assignment).  
**Hipótese:** A estrutura de sequências deve refletir as características vistas na EDA: Assignment A1 tem o maior número de estudantes com sequências completas; comprimentos médios variam entre assignments. A flag `is_first_attempt` deve cobrir todos os pares (SubjectID, ProblemID) únicos — um por estudante por problema.  
**Referência:** Shi et al. (2022) — Section 3 "Sequence truncation at 50"; Pankiewicz, Shi & Baker (2025).

In [6]:
from data_loader import build_sequences

# AssignmentIDs reais no dataset (não são 1–5 sequenciais)
assignment_ids = sorted(bkt_dkt_train['AssignmentID'].dropna().unique().tolist())
print(f'AssignmentIDs disponíveis em Release/Train: {assignment_ids}')

# Construir sequências para BKT/DKT — primeiro assignment como exemplo de verificação
a1_id = assignment_ids[0]
seqs_bkt_a1 = build_sequences(bkt_dkt_train, assignment_id=a1_id)

print(f'\nBKT/DKT — Release/Train — AssignmentID={a1_id}')
print(f'  Número de sequências (estudantes): {len(seqs_bkt_a1)}')

seq_lengths = [len(s['events']) for s in seqs_bkt_a1]
print(f'  Comprimento mínimo: {min(seq_lengths)}')
print(f'  Comprimento médio:  {sum(seq_lengths)/len(seq_lengths):.1f}')
print(f'  Comprimento máximo: {max(seq_lengths)}')

# Verificar is_first_attempt: deve ter exatamente 1 True por (SubjectID, ProblemID)
for seq in seqs_bkt_a1:
    df_s = seq['events']
    first_counts = df_s.groupby('ProblemID')['is_first_attempt'].sum()
    assert (first_counts == 1).all(), (
        f"Estudante {seq['subject_id']}: is_first_attempt inválido para problemas {first_counts[first_counts != 1].index.tolist()}"
    )
print('  OK — is_first_attempt: exatamente 1 True por (SubjectID, ProblemID)')

# Verificar ordenação cronológica dentro de cada sequência
for seq in seqs_bkt_a1:
    ts = seq['events']['ServerTimestamp']
    assert ts.is_monotonic_increasing, f"Sequência de {seq['subject_id']} não está ordenada cronologicamente"
print('  OK — ordenação cronológica verificada em todas as sequências')

# --- Exemplo de sequência (primeiros 6 eventos do primeiro estudante) ---
print()
print(f'=== Exemplo: primeiros 6 eventos da sequência do estudante 0 (A={a1_id}) ===')
example = seqs_bkt_a1[0]
print(f"subject_id:    {example['subject_id']}")
print(f"assignment_id: {example['assignment_id']}")
display_cols = ['ProblemID', 'ServerTimestamp', 'EventType', 'correct', 'is_first_attempt']
print(example['events'][display_cols].head(6).to_string(index=False))

# --- Contagem de sequências por assignment (BKT/DKT e Code-DKT) ---
print()
print(f'{"AssignmentID":<14} {"BKT/DKT seqs":>14} {"Code-DKT seqs":>15}')
print('-' * 46)
for aid in assignment_ids:
    n_bkt = len(build_sequences(bkt_dkt_train, aid))
    n_code = len(build_sequences(code_dkt_train, aid))
    print(f'{aid:<14} {n_bkt:>14,} {n_code:>15,}')

# --- Verificar que is_first_attempt cobre todos os pares únicos (SubjectID, ProblemID) ---
all_first = sum(
    seq['events']['is_first_attempt'].sum()
    for seq in seqs_bkt_a1
)
expected_pairs = bkt_dkt_train[bkt_dkt_train['AssignmentID'] == a1_id] \
    .groupby(['SubjectID', 'ProblemID']).ngroups
assert all_first == expected_pairs, (
    f"is_first_attempt count ({all_first}) != unique (SubjectID, ProblemID) pairs ({expected_pairs})"
)
print(f'\nOK — is_first_attempt cobre exatamente {expected_pairs} pares únicos (SubjectID, ProblemID) em A={a1_id}')

AssignmentIDs disponíveis em Release/Train: [439, 487, 492, 494, 502]



BKT/DKT — Release/Train — AssignmentID=439
  Número de sequências (estudantes): 233
  Comprimento mínimo: 5
  Comprimento médio:  37.6
  Comprimento máximo: 155
  OK — is_first_attempt: exatamente 1 True por (SubjectID, ProblemID)
  OK — ordenação cronológica verificada em todas as sequências

=== Exemplo: primeiros 6 eventos da sequência do estudante 0 (A=439) ===
subject_id:    04c32d4d95425f73b3a1d6502aed4d48
assignment_id: 439
 ProblemID           ServerTimestamp   EventType  correct  is_first_attempt
        13 2019-02-23 21:19:38+00:00 Run.Program        0              True
        13 2019-02-23 21:21:01+00:00 Run.Program        1             False
       232 2019-02-23 21:27:25+00:00 Run.Program        0              True
       232 2019-02-23 21:27:33+00:00 Run.Program        0             False
       232 2019-02-23 21:27:56+00:00 Run.Program        0             False
       232 2019-02-23 21:30:30+00:00 Run.Program        0             False

AssignmentID     BKT/DKT seqs  

439                       233             233


487                       224             224
492                       234             234


494                       221             221


502                       222             222

OK — is_first_attempt cobre exatamente 2271 pares únicos (SubjectID, ProblemID) em A=439


**Achado:** `build_sequences` constrói as sequências corretamente — a flag `is_first_attempt` cobre exatamente um evento por (SubjectID, ProblemID), a ordenação cronológica é garantida em todas as sequências, e o número de sequências varia por assignment (A1 com mais estudantes que A4/A5, refletindo o padrão de desistência visto na EDA). Estrutura de cada dicionário de sequência:

| Campo | Tipo | Descrição |
|-------|------|-----------|
| `subject_id` | `str` | Identificador anônimo do estudante |
| `assignment_id` | `int` | ID do assignment (1–5) |
| `events` | `pd.DataFrame` | Tentativas ordenadas por `ServerTimestamp` |

Colunas do DataFrame `events`:

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `SubjectID` | `str` | Identificador do estudante (redundante com `subject_id`) |
| `AssignmentID` | `Int64` | ID do assignment |
| `ProblemID` | `Int64` | ID do problema — KC do modelo |
| `ServerTimestamp` | `datetime64[UTC]` | Timestamp UTC da submissão |
| `EventType` | `str` | `'Run.Program'` (BKT/DKT) ou `{'Run.Program', 'Compile.Error'}` (Code-DKT) |
| `correct` | `int` | Label binária: 1 se `Run.Program` com `Score==1.0`, 0 caso contrário |
| `CodeStateID` | `str` | Referência ao estado do código (para extrair AST no Code-DKT) |
| `is_first_attempt` | `bool` | **True** para a primeira tentativa do estudante no problema (cronologicamente) |

**Implicação para modelagem:** A flag `is_first_attempt` permite calcular *first-attempt AUC* filtrando `events[events['is_first_attempt']]` após obter as predições do modelo — exatamente o protocolo de Shi et al. (2022). A estrutura em lista de dicionários (uma entrada por estudante) é diretamente iterável pelo loop de treinamento dos modelos DKT e Code-DKT, e convertível para os formatos esperados por pyBKT.

---

## 4 — Truncagem e Validação

### 4.1 — truncate_sequences()

**Contexto:** Shi et al. (2022) truncam cada sequência de estudante nas **últimas 50 tentativas** para limitar o custo computacional do LSTM e reduzir o viés de estudantes com históricos muito longos (outliers de engajamento). A truncagem mantém os eventos mais recentes porque eles refletem o estado de conhecimento mais próximo do momento de avaliação — e porque a maioria dos estudantes tem sequências menores que 50 (EDA: mediana ~37 tentativas em A1 BKT/DKT), tornando a truncagem efetiva apenas para os casos extremos.  
**Hipótese:** A maior parte das sequências (>50%) não será truncada; apenas os estudantes mais engajados (sequências longas) serão afetados. Após a truncagem, a taxa de corretos **deve aumentar** em relação à sequência completa, pois os eventos mais recentes correspondem à fase mais avançada do aprendizado — onde os estudantes erram menos. Para Code-DKT, o efeito deve ser ainda mais pronunciado porque os Compile.Error são mais frequentes no início da sequência.  
**Referência:** Shi et al. (2022) — Section 3, "We truncate student sequences to the last 50 attempts".

In [7]:
from data_loader import truncate_sequences

MAX_LEN = 50

# --- Construir sequências completas para todos os assignments ---
seqs_bkt_all  = {}  # assignment_id -> list[dict]
seqs_code_all = {}

for aid in assignment_ids:
    seqs_bkt_all[aid]  = build_sequences(bkt_dkt_train,  assignment_id=aid)
    seqs_code_all[aid] = build_sequences(code_dkt_train, assignment_id=aid)

# --- Truncar ---
seqs_bkt_trunc  = {aid: truncate_sequences(seqs, MAX_LEN) for aid, seqs in seqs_bkt_all.items()}
seqs_code_trunc = {aid: truncate_sequences(seqs, MAX_LEN) for aid, seqs in seqs_code_all.items()}

# --- Assertion principal: nenhuma sequência pode ter mais de MAX_LEN eventos ---
for aid, seqs in seqs_bkt_trunc.items():
    assert all(len(seq['events']) <= MAX_LEN for seq in seqs), \
        f"BKT/DKT A={aid}: sequência com mais de {MAX_LEN} eventos após truncagem"
for aid, seqs in seqs_code_trunc.items():
    assert all(len(seq['events']) <= MAX_LEN for seq in seqs), \
        f"Code-DKT A={aid}: sequência com mais de {MAX_LEN} eventos após truncagem"
print(f"OK — assert all(len(seq['events']) <= {MAX_LEN} ...) passa em todos os assignments e modelos")

# --- Estatísticas de truncagem por assignment ---
print()
print("=== BKT/DKT ===")
print(f"{'AssignmentID':<14} {'Seqs':>6} {'Truncadas':>10} {'% trunc':>9} "
      f"{'Len média (antes)':>18} {'Len média (depois)':>19}")
print('-' * 82)

for aid in assignment_ids:
    full  = seqs_bkt_all[aid]
    trunc = seqs_bkt_trunc[aid]
    n_trunc = sum(1 for s in full if len(s['events']) > MAX_LEN)
    avg_before = sum(len(s['events']) for s in full)  / len(full)
    avg_after  = sum(len(s['events']) for s in trunc) / len(trunc)
    print(f"{aid:<14} {len(full):>6} {n_trunc:>10} {n_trunc/len(full):>9.1%} "
          f"{avg_before:>18.1f} {avg_after:>19.1f}")

print()
print("=== Code-DKT ===")
print(f"{'AssignmentID':<14} {'Seqs':>6} {'Truncadas':>10} {'% trunc':>9} "
      f"{'Len média (antes)':>18} {'Len média (depois)':>19}")
print('-' * 82)

for aid in assignment_ids:
    full  = seqs_code_all[aid]
    trunc = seqs_code_trunc[aid]
    n_trunc = sum(1 for s in full if len(s['events']) > MAX_LEN)
    avg_before = sum(len(s['events']) for s in full)  / len(full)
    avg_after  = sum(len(s['events']) for s in trunc) / len(trunc)
    print(f"{aid:<14} {len(full):>6} {n_trunc:>10} {n_trunc/len(full):>9.1%} "
          f"{avg_before:>18.1f} {avg_after:>19.1f}")

# --- Taxa de corretos: antes vs depois da truncagem ---
print()
print("=== Taxa de corretos: antes vs depois da truncagem (Release/Train, todos os assignments) ===")

def correct_rate_from_seqs(seqs_by_aid):
    all_correct = sum(
        seq['events']['correct'].sum()
        for seqs in seqs_by_aid.values() for seq in seqs
    )
    all_total = sum(
        len(seq['events'])
        for seqs in seqs_by_aid.values() for seq in seqs
    )
    return all_correct / all_total if all_total > 0 else float('nan')

rate_bkt_before  = correct_rate_from_seqs(seqs_bkt_all)
rate_bkt_after   = correct_rate_from_seqs(seqs_bkt_trunc)
rate_code_before = correct_rate_from_seqs(seqs_code_all)
rate_code_after  = correct_rate_from_seqs(seqs_code_trunc)

print(f"{'Modelo':<14} {'Antes trunc':>14} {'Após trunc':>12} {'Δ':>8}")
print('-' * 52)
print(f"{'BKT/DKT':<14} {rate_bkt_before:>14.4%} {rate_bkt_after:>12.4%} {rate_bkt_after - rate_bkt_before:>+8.4%}")
print(f"{'Code-DKT':<14} {rate_code_before:>14.4%} {rate_code_after:>12.4%} {rate_code_after - rate_code_before:>+8.4%}")

# --- Verificar is_first_attempt após truncagem ---
for aid, seqs in seqs_bkt_trunc.items():
    for seq in seqs:
        events = seq['events']
        # Deve haver exatamente 1 is_first_attempt por ProblemID na janela truncada
        first_counts = events.groupby('ProblemID')['is_first_attempt'].sum()
        assert (first_counts == 1).all(), (
            f"BKT/DKT A={aid} subject={seq['subject_id']}: "
            f"is_first_attempt inválido após truncagem"
        )
print()
print("OK — is_first_attempt recalculado corretamente após truncagem em BKT/DKT")


OK — assert all(len(seq['events']) <= 50 ...) passa em todos os assignments e modelos

=== BKT/DKT ===
AssignmentID     Seqs  Truncadas   % trunc  Len média (antes)  Len média (depois)
----------------------------------------------------------------------------------
439               233         56     24.0%               37.6                31.8
487               224         81     36.2%               47.0                35.8
492               234         92     39.3%               51.2                33.6
494               221         54     24.4%               38.8                32.8
502               222         38     17.1%               31.4                26.8

=== Code-DKT ===
AssignmentID     Seqs  Truncadas   % trunc  Len média (antes)  Len média (depois)
----------------------------------------------------------------------------------
439               233        134     57.5%               86.0                40.6
487               224        151     67.4%               


OK — is_first_attempt recalculado corretamente após truncagem em BKT/DKT


**Achado:** A assertion `all(len(seq['events']) <= 50 ...)` passa sem erro em todos os assignments e modelos. A proporção de sequências efetivamente truncadas é baixa em BKT/DKT (a maioria dos estudantes tem <50 Run.Program); em Code-DKT a truncagem afeta mais sequências porque Compile.Error events são adicionados, aumentando o comprimento médio. A taxa de corretos **aumenta após a truncagem** em BKT/DKT (os eventos mais recentes refletem o aprendizado acumulado — estudantes erram menos no final). Para Code-DKT, o efeito é menor porque Compile.Error events (sempre `correct=0`) aparecem ao longo de toda a sequência, e a janela final ainda pode conter erros de compilação.

**Divergência em relação ao valor bruto:** A taxa bruta de Release/Train em Run.Program é 23.70% (benchmark Shi et al., 2022). Após truncagem, a taxa sobe levemente porque eventos do início da sequência — onde a taxa de acertos é menor (fase de familiarização com o problema) — são removidos. Essa divergência é esperada e desejável: o modelo é treinado na janela mais recente (estado atual de conhecimento), não no histórico completo.  
**Implicação para modelagem:** A truncagem não distorce a métrica *first-attempt AUC* (calculada sobre `is_first_attempt`, que é recalculado corretamente na janela truncada). O efeito principal é reduzir o custo de memória do LSTM (sequências de comprimento fixo ≤50) e eliminar o viés de estudantes com históricos muito longos que dominariam o gradiente.

---

## 5 — Serialização dos Artefatos

### 5.1 — Salvar sequências processadas em disco

**Contexto:** Após filtragem, construção e truncagem, as sequências KT precisam ser serializadas em disco para que os notebooks de modelagem (04–06) possam carregá-las diretamente sem re-executar o pipeline de pré-processamento. A serialização também serve como ponto de reprodutibilidade: qualquer colaborador pode re-executar `02_preprocessing.ipynb` e obter artefatos idênticos (SEED=42 fixado).
**Hipótese:** Dois artefatos distintos são necessários — `sequences_bkt_dkt.pkl` e `sequences_code_dkt.pkl` — porque os protocolos de filtragem diferem: BKT/DKT usam apenas `Run.Program`; Code-DKT inclui `Compile.Error`. Cada artefato armazena os splits `train` e `test` já truncados (máx. 50 eventos/sequência), indexados por `AssignmentID`.
**Referência:** Shi et al. (2022) — protocolo de split e truncagem; Pankiewicz, Shi & Baker (2025) — inclusão de Compile.Error no Code-DKT.

In [8]:
import pickle

# --- Construir e truncar sequências de teste ---
seqs_bkt_test_all  = {aid: build_sequences(bkt_dkt_test,  assignment_id=aid) for aid in assignment_ids}
seqs_code_test_all = {aid: build_sequences(code_dkt_test, assignment_id=aid) for aid in assignment_ids}

seqs_bkt_test_trunc  = {aid: truncate_sequences(seqs, MAX_LEN) for aid, seqs in seqs_bkt_test_all.items()}
seqs_code_test_trunc = {aid: truncate_sequences(seqs, MAX_LEN) for aid, seqs in seqs_code_test_all.items()}

# Nota: Release/Test tem apenas 3 assignments (439, 487, 492) — A494 e A502 retornam listas vazias
test_aids_bkt  = [aid for aid, seqs in seqs_bkt_test_trunc.items()  if seqs]
test_aids_code = [aid for aid, seqs in seqs_code_test_trunc.items() if seqs]
print(f'AssignmentIDs com sequências em Release/Test — BKT/DKT: {test_aids_bkt}')
print(f'AssignmentIDs com sequências em Release/Test — Code-DKT: {test_aids_code}')

# --- Montar artefatos ---
artifact_bkt_dkt = {
    'train':          seqs_bkt_trunc,
    'test':           seqs_bkt_test_trunc,
    'assignment_ids': assignment_ids,
    'max_len':        MAX_LEN,
    'seed':           SEED,
    'description':    'Sequências KT para BKT e DKT — apenas Run.Program, correct=(Score==1.0)',
}

artifact_code_dkt = {
    'train':          seqs_code_trunc,
    'test':           seqs_code_test_trunc,
    'assignment_ids': assignment_ids,
    'max_len':        MAX_LEN,
    'seed':           SEED,
    'description':    'Sequências KT para Code-DKT — Run.Program + Compile.Error (correct=0)',
}

# --- Serializar ---
path_bkt  = RESULTS_DIR / 'sequences_bkt_dkt.pkl'
path_code = RESULTS_DIR / 'sequences_code_dkt.pkl'

with open(path_bkt, 'wb') as f:
    pickle.dump(artifact_bkt_dkt, f, protocol=pickle.HIGHEST_PROTOCOL)
with open(path_code, 'wb') as f:
    pickle.dump(artifact_code_dkt, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f'\nArtefatos salvos em {RESULTS_DIR}/')
print(f'  sequences_bkt_dkt.pkl  — {path_bkt.stat().st_size / 1024:.1f} KB')
print(f'  sequences_code_dkt.pkl — {path_code.stat().st_size / 1024:.1f} KB')

# --- Estatísticas detalhadas ---
def artifact_stats(artifact, label):
    print(f'\n=== {label} ===')
    for split_name in ('train', 'test'):
        seqs_by_aid = artifact[split_name]
        all_lengths = [len(seq['events']) for seqs in seqs_by_aid.values() for seq in seqs]
        if not all_lengths:
            print(f'  {split_name}: sem sequências')
            continue
        total_seqs   = sum(len(seqs) for seqs in seqs_by_aid.values())
        total_events = sum(all_lengths)
        all_correct  = sum(
            seq['events']['correct'].sum()
            for seqs in seqs_by_aid.values() for seq in seqs
        )
        aids_with_data = [aid for aid, seqs in seqs_by_aid.items() if seqs]
        unique_students = len({seq['subject_id'] for seqs in seqs_by_aid.values() for seq in seqs})
        print(f'  {split_name}:')
        print(f'    Assignments c/ dados:  {len(aids_with_data)} ({aids_with_data})')
        print(f'    Estudantes únicos:     {unique_students:,}')
        print(f'    Sequências total:      {total_seqs:,}')
        print(f'    Comprimento mín:       {min(all_lengths)}')
        print(f'    Comprimento médio:     {sum(all_lengths)/len(all_lengths):.1f}')
        print(f'    Comprimento máx:       {max(all_lengths)}')
        print(f'    Eventos total:         {total_events:,}')
        print(f'    Taxa de corretos:      {all_correct/total_events:.4%}')
        print(f'    Por assignment:')
        print(f'    {"AID":<8} {"Seqs":>6} {"Len min":>8} {"Len méd":>9} {"Len máx":>8}')
        print(f'    {"-"*46}')
        for aid, seqs in seqs_by_aid.items():
            if not seqs:
                print(f'    {aid:<8} {0:>6} {"—":>8} {"—":>9} {"—":>8}')
                continue
            lens = [len(s['events']) for s in seqs]
            print(f'    {aid:<8} {len(seqs):>6} {min(lens):>8} {sum(lens)/len(lens):>9.1f} {max(lens):>8}')

artifact_stats(artifact_bkt_dkt,  'BKT/DKT')
artifact_stats(artifact_code_dkt, 'Code-DKT')

# --- Schema keys ---
print('\n=== Schema dos artefatos (chaves de nível superior) ===')
for k, v in artifact_bkt_dkt.items():
    if isinstance(v, dict):
        print(f'  {k!r:<20} dict[int -> list[dict]]  # {len(v)} assignments')
    elif isinstance(v, list):
        print(f'  {k!r:<20} list[int]  # {v}')
    else:
        print(f'  {k!r:<20} {type(v).__name__}  # {v!r}')

print('\n=== Schema de cada sequência (dict interno) ===')
example_seq = artifact_bkt_dkt['train'][assignment_ids[0]][0]
for k, v in example_seq.items():
    if hasattr(v, 'dtypes'):
        print(f'  {k!r:<20} pd.DataFrame  # {len(v)} linhas, colunas: {list(v.columns)}')
    else:
        print(f'  {k!r:<20} {type(v).__name__}  # {v!r}')


AssignmentIDs com sequências em Release/Test — BKT/DKT: [439, 487, 492]
AssignmentIDs com sequências em Release/Test — Code-DKT: [439, 487, 492]



Artefatos salvos em /home/leokuntz/Documents/repositories/studies/tcc.edm.kt/results/
  sequences_bkt_dkt.pkl  — 9613.5 KB
  sequences_code_dkt.pkl — 10617.8 KB

=== BKT/DKT ===
  train:
    Assignments c/ dados:  5 ([439, 487, 492, 494, 502])
    Estudantes únicos:     246
    Sequências total:      1,134
    Comprimento mín:       1
    Comprimento médio:     32.2
    Comprimento máx:       50
    Eventos total:         36,497
    Taxa de corretos:      27.9694%
    Por assignment:
    AID        Seqs  Len min   Len méd  Len máx
    ----------------------------------------------
    439         233        5      31.8       50
    487         224        2      35.8       50
    492         234        8      33.6       50
    494         221        1      32.8       50
    502         222        1      26.8       50
  test:
    Assignments c/ dados:  3 ([439, 487, 492])
    Estudantes únicos:     83
    Sequências total:      236
    Comprimento mín:       3
    Comprimento médio:    

**Achado:** Os dois artefatos foram serializados com sucesso — `sequences_bkt_dkt.pkl` (9.4 MB) e `sequences_code_dkt.pkl` (10.4 MB). Estatísticas consolidadas:

| Artefato | Split | Estudantes | Sequências | Comprimento mín | Comprimento médio | Comprimento máx | Taxa corretos |
|----------|-------|-----------|-----------|-----------------|-------------------|-----------------|---------------|
| BKT/DKT | train | 246 | 1.134 (5 assignments) | 1 | 32.2 | 50 | 27.97% |
| BKT/DKT | test  | 83  | 236 (3 assignments)   | 3 | 33.3 | 50 | 26.70% |
| Code-DKT | train | 246 | 1.134 (5 assignments) | 1 | 39.2 | 50 | 19.87% |
| Code-DKT | test  | 83  | 236 (3 assignments)   | 4 | 41.5 | 50 | 18.94% |

**Nota sobre Release/Test:** apenas 3 assignments têm sequências de teste (A439, A487, A492); A494 e A502 têm 0 sequências em Release/Test — comportamento consistente com Shi et al. (2022), que avalia apenas os assignments presentes no split de teste. O artefato preserva todos os 5 AssignmentIDs para compatibilidade, mas as listas de teste de A494 e A502 são vazias.

**Schema dos artefatos `sequences_bkt_dkt.pkl` e `sequences_code_dkt.pkl`:**

| Chave | Tipo | Descrição |
|-------|------|-----------|
| `'train'` | `dict[int → list[dict]]` | Sequências truncadas de treino, indexadas por `AssignmentID` |
| `'test'` | `dict[int → list[dict]]` | Sequências truncadas de teste, indexadas por `AssignmentID` |
| `'assignment_ids'` | `list[int]` | IDs dos 5 assignments: `[439, 487, 492, 494, 502]` |
| `'max_len'` | `int` | Comprimento máximo das sequências: `50` |
| `'seed'` | `int` | Seed de reprodutibilidade: `42` |
| `'description'` | `str` | Texto descritivo do artefato |

Cada sequência (item da lista interna) é um dicionário com:

| Chave | Tipo | Descrição |
|-------|------|-----------|
| `'subject_id'` | `str` | ID anônimo do estudante |
| `'assignment_id'` | `int` | ID do assignment |
| `'events'` | `pd.DataFrame` | Eventos ordenados cronologicamente, máx. 50 linhas |

Colunas de `events`: `SubjectID`, `AssignmentID`, `ProblemID`, `ServerTimestamp`, `EventType`, `correct` (int 0/1), `CodeStateID`, `is_first_attempt` (bool), além de colunas auxiliares (`Score`, `Compile.Result`, etc.).

**Implicação para modelagem:** Os notebooks de modelagem (04–06) carregam os artefatos com `pickle.load` e iteram diretamente sobre as listas de sequências. A flag `is_first_attempt` está preservada no `events` DataFrame, permitindo calcular *first-attempt AUC* sem re-executar o pré-processamento. O artefato `sequences_code_dkt.pkl` inclui eventos `Compile.Error` com `correct=0`, prontos para extração de features AST via srcML no notebook `03_code_features.ipynb`.